# 第2回 宿題（基礎）: `lora_r` を変えて学習し、比較する

`lora_format_tuning.ipynb` と同じ題材・同じ設定で、LoRA の行列の次元 `lora_r` だけを 2, 4, 8, 16, 32 と変えて学習し、次の 2 点を比べます。

1. 学習と評価: 学習パラメータ数、損失の推移、形式遵守率（`format_rate`）と内容正解率（`content_rate`）、残った失敗、知識を問う質問の分類。
2. 学習された更新 $\Delta W = \frac{\alpha}{r} B A$ の特異値: 更新が少数の方向に集中しているか、r を変えても同じ方向が学習されるか、$A$ が乱数による初期値からどれだけ動いたか。

**進め方**

| セクション | 内容 | 所要時間の目安（T4） |
| --- | --- | --- |
| 0〜3 | 環境、設定、データ生成、モデル読み込み、学習前評価（`lora_format_tuning.ipynb` のセクション 0〜3 と同じコード） | 約 5 分 |
| 4 | r を変えて学習・評価し、特異値を記録する（5 条件） | 約 25〜30 分 |
| 4b | 集計（表を表示） | 1 分未満 |

**使うもの**: Google Colab（T4 GPU）。ランタイムのタイプで T4 GPU を選んでから、上から順に実行してください。終わったら **ランタイム → ランタイムを接続解除して削除** を忘れずに。

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定

`lora_format_tuning.ipynb` のセクション 1 と同じ設定です。`lora_r` はセクション 4 で条件ごとに指定するので、ここの `lora_r: 16` は使いません。

**見るところ**: 設定の JSON が表示されます。

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 20,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


## 2. 架空 FAQ と教師データの生成

`lora_format_tuning.ipynb` のセクション 2 と同じデータ（train 14 件、valid 7 件、test 21 問）を作ります。

**見るところ**: `train=14 valid=7 test=21 gold=21` と表示されること。

In [4]:
FAQ = {
    "deadline": "課題の提出期限は 9 月 30 日 23:59（日本時間）です。",
    "destination": "提出先は学習ポータルの「課題」欄です。",
    "format": "提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。",
    "office_hours": "質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。",
    "late": "期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

# 根拠あり: (FAQ キー, 質問テンプレート群, 正解の answer に含まれるべき語)
GROUNDED = {
    "deadline": (["課題の提出期限はいつですか", "いつまでに出せばいいですか", "締切を教えてください", "提出の期限は？"], ["9 月 30 日", "23:59"]),
    "destination": (["提出先はどこですか", "どこにアップロードしますか", "課題はどこから出しますか", "提出場所は？"], ["学習ポータル", "課題"]),
    "format": (["提出形式を教えてください", "ファイル名の決まりはありますか", "何のファイル形式で出しますか", "PDF 以外でも出せますか"], ["PDF"]),
    "office_hours": (["質問はいつできますか", "オフィスアワーはいつですか", "先生に質問する時間はありますか", "相談できる時間帯は？"], ["火曜", "17:00"]),
    "late": (["遅れて出したらどうなりますか", "期限を過ぎた場合は受け付けますか", "遅延提出のルールは？", "締切後の扱いを教えてください"], ["48 時間", "減点"]),
}
# 根拠なし: FAQ に無いので needs_review=true が正解
UNGROUNDED = [
    "再提出はできますか", "欠席した回の扱いを教えてください", "他の受講者の点数は見られますか",
    "成績はいつ出ますか", "提出期限は延長できますか", "グループでの提出は可能ですか",
    "課題の配点はどうなっていますか", "講義の録画は公開されますか",
]

# test 専用の追加質問（train / valid には入れない）
TEST_EXTRA_GROUNDED = {
    "deadline": ["提出期限の日付と時刻を教えてください", "課題の締切は何月何日の何時ですか"],
    "destination": ["課題の提出はどこで行いますか", "提出物を出す先を教えてください"],
    "format": ["提出するファイルの形式は何ですか", "提出ファイルは何形式で、いくつ出しますか"],
    "office_hours": ["オフィスアワーの曜日と時間を教えてください", "質問を受け付けてもらえるのは何曜日の何時からですか"],
    "late": ["提出が締切に間に合わなかったらどうなりますか", "期限後に提出した課題は受理されますか"],
}
TEST_EXTRA_UNGROUNDED = [
    "課題の文字数に制限はありますか", "提出した課題へのフィードバックはもらえますか",
    "参考文献の書き方に指定はありますか", "オフィスアワーは事前予約が必要ですか",
]
UNGROUNDED_ANSWER = "この FAQ には記載がありません。運営に確認が必要です。"

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # テンプレートの先頭 2 つを train、3 つ目を valid、4 つ目を test に固定（言い換えを跨がせない）
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # 根拠なしの質問は 4 問ごとに train 2 / valid 1 / test 1 に振る
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["確認"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["確認"]}

# train は 14 件（根拠あり 10、根拠なし 4）。同じ例を何回学習するかは、データを複製せず、セクション 4 の num_train_epochs で決める
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "質問が train / valid / test のどこかで重複しています。テンプレートを見直してください"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("学習例の 1 件 | 質問:", train[0]["question"], "| 望ましい出力:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

train=14 valid=7 test=21 gold=21
学習例の 1 件 | 質問: 課題の提出期限はいつですか | 望ましい出力: {"answer": "課題の提出期限は 9 月 30 日 23:59（日本時間）です。", "needs_review": false}


## 3. モデルの読み込みと学習前評価

`lora_format_tuning.ipynb` のセクション 3 と同じです。学習前の評価は全条件で共通なので、ここで 1 回だけ行います。

**見るところ**: `重みのメモリ(GB): 3.01` と、学習前の 1 行要約（`format_rate` など）と 21 問の表。

**出ても問題ない警告**: `You are sending unauthenticated requests to the HF Hub`、`... is falling back to its reference PyTorch implementation ...`（`lora_format_tuning.ipynb` のセクション 3 の説明を参照）。

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 形式はプロンプトで指示している。モデルがこれを守れるかが学習前の確認点
SYSTEM = "あなたは講座 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。FAQ に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [6]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def check_format(text: str):
    '''JSON として読め、必須キーと型が合うか。失敗理由も返す'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''内容の合否: needs_review が正解と一致し、必須語句を answer に含む'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''test の 21 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は FAQ と合っているかを読む
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 21, "format_rate": 0.09523809523809523, "content_rate": 0.0, "both_rate": 0.0, "seconds": 19.3}


,質問,正解の needs_review,学習前の出力,形式,形式の失敗理由,内容
0,提出の期限は？,False,"{""answer"": ""9 月 30 日 23:59""}",×,missing_key,×
1,提出期限の日付と時刻を教えてください,False,"{""answer"": ""9 月 30 日 23:59""}",×,missing_key,×
2,課題の締切は何月何日の何時ですか,False,"{""answer"": ""9 月 30 日 23:59""}",×,missing_key,×
3,提出場所は？,False,"{""answer"": ""学習ポータルの「課題」欄""}",×,missing_key,×
4,課題の提出はどこで行いますか,False,"{""answer"": ""学習ポータルの「課題」欄で提出先を指定し、PDF 1 ファイルで提出します。提出時間は毎週火曜 17:00〜18:00 のオフィスアワーで行います。""}",×,missing_key,×
5,提出物を出す先を教えてください,False,"{""answer"": ""学習ポータルの「課題」欄""}",×,missing_key,×
6,PDF 以外でも出せますか,False,"{""answer"": ""確認が必要""}",×,missing_key,×
7,提出するファイルの形式は何ですか,False,"{""answer"": ""PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です""}",×,missing_key,×
8,提出ファイルは何形式で、いくつ出しますか,False,"{""answer"": ""PDF 1 ファイルで、1 出します""}",×,missing_key,×
9,相談できる時間帯は？,False,"{""answer"": ""火曜 17:00〜18:00 のオフィスアワー""}",×,missing_key,×


## 4. `lora_r` を変えて学習・評価し、特異値を記録する

**やること**: `lora_r` だけを 2, 4, 8, 16, 32 と変えて学習し、学習後の評価（`lora_format_tuning.ipynb` のセクション 5、6 と同じ採点）と、各 LoRA 層の更新 $\frac{\alpha}{r}BA$ の特異値を記録します。`lora_alpha` は 32 のまま固定するので、更新にかかる倍率 $\alpha / r$ も 16, 8, 4, 2, 1 と同時に変わります。

- 条件ごとに基盤モデルを読み込み直すので、前の条件の adapter は残りません。セクション 3 で読み込んだ `model` は、最初に GPU から外します。
- r = 16 も、ほかの r と同じ手順で回します（学習前の $A$ の記録と、条件をそろえるため）。
- 学習の設定は `lora_format_tuning.ipynb` のセクション 4 と同じです。違いは次の 3 点で、いずれも学習の計算そのものは変えません。
  - 学習の直前に `transformers.set_seed` を呼ぶ（LoRA の初期値を条件ごとに再現できるようにする）。
  - `eval_strategy="steps"` で、valid の損失を 10 ステップごとに記録する。
  - 途中経過を保存しない（`save_strategy="no"`）。
- 結果は条件ごとに `/content/qlora_lab/rank_sweep/` に保存されます。途中で止まっても、このセルを実行し直せば、終わった条件は飛ばします（ランタイムごと消えた場合を除く）。`USE_DRIVE = True` なら Drive にもコピーします。

**見るところ**: 条件ごとに `trainable params`、学習の進み具合、学習後の評価の 1 行要約、`r=...: 学習 ... 秒、合計 ... 秒` が表示されます。`trainable params` は r に比例します（r = 16 で 6,389,760）。

**所要時間の目安（T4）**: 1 条件あたり約 5〜6 分（学習約 4 分と評価）、5 条件で約 25〜30 分です。Colab の無料枠の GPU 時間に注意してください。

In [7]:
# 前提: セクション 0〜3 を実行済み（CONFIG, WORK, DEVICE, tok, train/valid/test/gold, make_example, build_messages, generate, check_format, evaluate, before_rows を使う）
import gc, shutil
import transformers
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

RANKS = [2, 4, 8, 16, 32]   # r=16 も、学習前の A を記録して条件をそろえるために回し直す
SWEEP = WORK / "rank_sweep"; SWEEP.mkdir(exist_ok=True)

# セクション 3 で読み込んだ基盤モデルを GPU から外す（条件ごとに読み込み直すため）
globals().pop("model", None)
gc.collect(); torch.cuda.empty_cache()

# lora_format_tuning.ipynb のセクション 4、6 と同じ定義
KNOWLEDGE_QUESTIONS = ["この講座の最終課題の締切はいつですか", "担当教員の名前は何ですか",
                       "次回の講義のテーマは何ですか", "課題の提出期限はいつですか"]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"
def to_chat(example):
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}
def knowledge(m):
    return [{"question": q, "raw": (raw := generate(m, make_example(q, "", True))), "class": classify(raw)} for q in KNOWLEDGE_QUESTIONS]
train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

def lora_weights(m):
    '''各 LoRA 層の (A, B) を CPU の float64 で返す'''
    return {name.removeprefix("base_model.model."): (mod.lora_A["default"].weight.detach().cpu().double(),
                                                    mod.lora_B["default"].weight.detach().cpu().double())
            for name, mod in m.named_modules() if hasattr(mod, "lora_A") and "default" in mod.lora_A}

def lora_svd(m, A_init, k_keep=8):
    '''(α/r)BA の特異値と上位 k_keep 個の特異ベクトル。BA = Q_B R_B R_A^T Q_A^T なので、r×r 行列 R_B R_A^T の特異値分解で足りる'''
    out = {}
    for name, (A, B) in lora_weights(m).items():
        s = float(m.get_submodule("base_model.model." + name).scaling["default"])
        QB, RB = torch.linalg.qr(B); QA, RA = torch.linalg.qr(A.T)
        Uc, S, Vch = torch.linalg.svd(s * RB @ RA.T)
        if not out:   # 最初の 1 層だけ、行列を直接作って特異値が一致することを確かめる
            assert torch.allclose(torch.linalg.svdvals(s * B @ A)[:S.numel()], S, rtol=1e-8, atol=1e-10)
        k = min(k_keep, S.numel()); V = QA @ Vch.T
        QA0, _ = torch.linalg.qr(A_init[name].T)
        out[name] = {"S": S, "U": (QB @ Uc)[:, :k].float(), "V": V[:, :k].float(),
                     "fro": float(S.square().sum().sqrt()), "shape": (B.shape[0], A.shape[1]),
                     "A_rel_change": float((A - A_init[name]).norm() / A_init[name].norm()),   # ‖A−A0‖_F / ‖A0‖_F
                     "A_rowspace_overlap": float((QA0.T @ QA).square().sum() / A.shape[0]),   # 学習前後の A の行空間の重なり（1 で一致）
                     "V_in_init_rowspace": (QA0.T @ V).square().sum(0).tolist()}              # ΔW の各右特異ベクトルが学習前の A の行空間に入る割合
    return out

for r in RANKS:
    if (SWEEP / f"r{r}.json").exists():
        print(f"r={r}: 結果があるので飛ばします"); continue
    print(f"===== lora_r = {r}（lora_alpha = {CONFIG['lora_alpha']}、倍率 α/r = {CONFIG['lora_alpha'] / r}） =====")
    t_start = time.time()
    base = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
    if not (SWEEP / "knowledge_before.json").exists():   # 学習前の知識質問は全条件で共通なので 1 回だけ
        (SWEEP / "knowledge_before.json").write_text(json.dumps(knowledge(base), ensure_ascii=False, indent=2), encoding="utf-8")
    base.config.use_cache = False
    transformers.set_seed(CONFIG["seed"])   # LoRA の初期値を条件ごとに再現できるようにする
    lora = LoraConfig(r=r, lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    sft = SFTConfig(output_dir=str(SWEEP / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                    learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10,
                    save_strategy="no", eval_strategy="steps", eval_steps=10,   # セクション 4 との違い: 途中保存なし、valid の損失を 10 ステップごとに記録
                    fp16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
    tr = SFTTrainer(model=base, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
    A_init = {k: a for k, (a, b) in lora_weights(tr.model).items()}   # 学習前の A（B は 0 で初期化されている）
    n_trainable = sum(p.numel() for p in tr.model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in tr.model.parameters())
    print(f"trainable params: {n_trainable:,} || all params: {n_all:,} || trainable%: {100 * n_trainable / n_all:.4f}")
    t0 = time.time(); tr.train(); train_sec = time.time() - t0
    m = tr.model; m.config.use_cache = True; _ = m.eval()
    m.save_pretrained(SWEEP / f"adapter_r{r}")
    torch.save(lora_svd(m, A_init), SWEEP / f"svd_r{r}.pt")
    s_after, rows_after = evaluate(m, f"after_r{r}")
    res = {"lora_r": r, "lora_alpha": CONFIG["lora_alpha"], "scaling": CONFIG["lora_alpha"] / r,
           "trainable_params": n_trainable, "all_params": n_all, "train_seconds": round(train_sec, 1),
           "steps": tr.state.global_step, "log_history": tr.state.log_history,
           "after": s_after, "after_rows": rows_after,
           "regressed": [b["question"] for b, a in zip(before_rows, rows_after) if b["both_ok"] and not a["both_ok"]],
           "knowledge": knowledge(m)}
    (SWEEP / f"r{r}.json").write_text(json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")
    if USE_DRIVE:
        shutil.copytree(SWEEP, Path(CONFIG["drive_dir"]) / "rank_sweep", dirs_exist_ok=True)
    del tr, m, base; gc.collect(); torch.cuda.empty_cache()
    print(f"r={r}: 学習 {train_sec:.0f} 秒、合計 {time.time() - t_start:.0f} 秒")

===== lora_r = 2（lora_alpha = 32、倍率 α/r = 16.0） =====


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

... (56 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 12,779,520 || all params: 765,172,544 || trainable%: 1.6701


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,1.567699,0.270814,0.531571,17860.000000,0.943322
20,0.087614,0.125372,0.101167,35720.000000,0.981481
30,0.020515,0.132857,0.071622,53580.000000,0.981481
40,0.015089,0.133514,0.072165,71440.000000,0.981481


{"label": "after_r32", "n": 21, "format_rate": 1.0, "content_rate": 0.9047619047619048, "both_rate": 0.9047619047619048, "seconds": 40.3}
r=32: 学習 229 秒、合計 287 秒


### 4b. 集計

**やること**: セクション 4 の結果（`/content/qlora_lab/rank_sweep/`）を集計し、Markdown の表として表示します。表はそのまま `answer.md` に貼れます。途中までしか終わっていなくても、終わった条件だけで集計します。

**表の見方**

1. 学習と評価: 学習パラメータ数、train と valid の損失の推移、`format_rate` と `content_rate`、内容が × の質問、知識質問の分類。
2. 特異値: 層の種類（注意機構 `attn`、MLP `mlp`）ごとの中央値です。各層の特異値を $\sigma_1 \ge \sigma_2 \ge \cdots \ge \sigma_r$ とし、上位 $k$ 個の特異値の 2 乗和を $S_k = \sum_{i=1}^{k} \sigma_i^2$、全体の 2 乗和を $S = S_r$ と書きます（$S = \|\Delta W\|_F^2$、全成分の 2 乗和）。
   - `σ1²/Σσ²` の列 $= S_1 / S$: 最大の特異値の方向が $S$ に占める割合。1 に近いほど、更新が 1 方向に集中している。
   - 安定ランク `Σσ²/σ1²` の列 $= S / S_1$: 実質的な階数の目安。r より小さければ、使える次元を使い切っていない。
   - `90% に要る個数` の列: $S_k / S \ge 0.9$ となる最小の $k$。
   - `‖ΔW‖_F（全層合計）` の列: 層ごとの $S$ を足し合わせたものの平方根 $\sqrt{\sum_{\text{層}} S}$。
3. $A$ が学習でどれだけ動いたか: 学習前の $A_0$ との比較。行空間の重なりが 1 に近ければ、入力側の方向は乱数による初期値でほぼ決まっている。
4. 異なる r の間の部分空間の重なり $\varphi = \|U_i^\top U'_j\|_F^2 / \min(i, j)$（LoRA の原論文 7.2 節と同じ指標）: 1 で一致、0 で直交。括弧内は、無作為な部分空間どうしの期待値 $\max(i, j) / d$ の中央値。

In [8]:
import itertools, statistics
med = statistics.median
ranks = sorted(int(p.stem[1:]) for p in SWEEP.glob("r*.json"))
res = {r: json.loads((SWEEP / f"r{r}.json").read_text(encoding="utf-8")) for r in ranks}
svd = {r: torch.load(SWEEP / f"svd_r{r}.pt") for r in ranks}
knowledge_before = json.loads((SWEEP / "knowledge_before.json").read_text(encoding="utf-8"))

def md_table(header, rows):
    print("| " + " | ".join(header) + " |"); print("| " + " | ".join("---" for _ in header) + " |")
    for row in rows: print("| " + " | ".join(str(c) for c in row) + " |")
    print()
def kind(name):
    return "attn" if any(k in name for k in ("q_proj", "k_proj", "v_proj", "o_proj")) else "mlp"

print("## 1. 学習と評価\n")
print(f"学習前: format_rate={before_summary['format_rate']:.3f}, content_rate={before_summary['content_rate']:.3f}\n")
rows = []
for r in ranks:
    R = res[r]
    tl = {e["step"]: e["loss"] for e in R["log_history"] if "loss" in e}
    el = {e["step"]: e["eval_loss"] for e in R["log_history"] if "eval_loss" in e}
    rows.append([r, R["scaling"], f"{R['trainable_params']:,}", f"{100 * R['trainable_params'] / R['all_params']:.4f}", R["train_seconds"],
                 " / ".join(f"{tl[s]:.4f}" for s in sorted(tl)), " / ".join(f"{el[s]:.4f}" for s in sorted(el)),
                 f"{R['after']['format_rate']:.3f}", f"{R['after']['content_rate']:.3f}", len(R["regressed"])])
md_table(["r", "α/r", "学習パラメータ", "trainable%", "学習秒", "train loss (10/20/30/40)", "valid loss (10/20/30/40)",
          "format_rate", "content_rate", "退行"], rows)

print("### 内容が × の質問\n")
for r in ranks:
    fails = [x for x in res[r]["after_rows"] if not x["both_ok"]]
    print(f"- r={r}: " + ("なし" if not fails else ""))
    for x in fails: print(f"  - {x['question']}（正解の needs_review={x['gold_needs_review']}）: `{x['raw']}`")
print()

print("### 知識を問う質問（学習前 → 各 r の学習後）\n")
md_table(["質問", "学習前"] + [f"r={r}" for r in ranks],
         [[k["question"], k["class"]] + [res[r]["knowledge"][i]["class"] for r in ranks] for i, k in enumerate(knowledge_before)])
for r in ranks:
    for k in res[r]["knowledge"]:
        if k["class"] == "asserted": print(f"- r={r} で断定: {k['question']} → `{k['raw']}`")
print()

print("## 2. (α/r)BA の特異値（層の種類ごとの中央値）\n")
rows = []
for r in ranks:
    for kd in ("attn", "mlp"):
        mods = [v for n, v in svd[r].items() if kind(n) == kd]
        Ss = [v["S"] for v in mods]
        rows.append([r, kd, len(mods),
                     f"{med(float(S[0]**2 / S.square().sum()) for S in Ss):.3f}",
                     f"{med(float(S.square().sum() / S[0]**2) for S in Ss):.2f}",
                     med(int((torch.cumsum(S.square(), 0) / S.square().sum() < 0.9).sum()) + 1 for S in Ss),
                     " ".join(f"{med(float(S[i] / S[0]) for S in Ss):.2f}" for i in range(min(r, 8))),
                     f"{sum(v['fro']**2 for v in mods)**0.5:.3f}"])
md_table(["r", "層", "層数", "σ1²/Σσ²", "安定ランク Σσ²/σ1²", "90% に要る個数", "σi/σ1 (i=1..min(r,8))", "‖ΔW‖_F（全層合計）"], rows)

print("## 3. A は学習でどれだけ動いたか（中央値）\n")
rows = []
for r in ranks:
    for kd in ("attn", "mlp"):
        mods = [v for n, v in svd[r].items() if kind(n) == kd]
        rows.append([r, kd, f"{med(v['A_rel_change'] for v in mods):.4f}", f"{med(v['A_rowspace_overlap'] for v in mods):.4f}",
                     f"{med(v['V_in_init_rowspace'][0] for v in mods):.4f}"])
md_table(["r", "層", "‖A−A0‖/‖A0‖", "学習前後の A の行空間の重なり", "ΔW の第 1 右特異ベクトルが学習前の A の行空間に入る割合"], rows)

def phi(X, Y, i, j):
    return float((X[:, :i].T @ Y[:, :j]).square().sum() / min(i, j))
print("## 4. 異なる r の間の部分空間の重なり φ（中央値。括弧内は無作為な部分空間どうしの期待値の中央値）\n")
rows = []
for r1, r2 in itertools.combinations(ranks, 2):
    for side, key, dim in (("出力側 U", "U", 0), ("入力側 V", "V", 1)):
        cells = []
        for i, j in ((1, 1), (2, 2), (1, min(r2, 8))):
            val = med(phi(svd[r1][n][key], svd[r2][n][key], i, j) for n in svd[r1])
            rnd = med(max(i, j) / svd[r1][n]["shape"][dim] for n in svd[r1])   # 無作為なら E‖X^T Y‖_F² = i j / d
            cells.append(f"{val:.3f} ({rnd:.4f})")
        rows.append([f"{r1} vs {r2}", side] + cells)
md_table(["比較", "側", "φ(1,1)", "φ(2,2)", "φ(1, min(r2,8))"], rows)

## 1. 学習と評価

学習前: format_rate=0.095, content_rate=0.000

| r | α/r | 学習パラメータ | trainable% | 学習秒 | train loss (10/20/30/40) | valid loss (10/20/30/40) | format_rate | content_rate | 退行 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2 | 16.0 | 798,720 | 0.1060 | 228.3 | 1.5307 / 0.0835 / 0.0164 / 0.0127 | 0.2636 / 0.1243 / 0.1338 / 0.1328 | 1.000 | 0.905 | 0 |
| 4 | 8.0 | 1,597,440 | 0.2119 | 228.5 | 1.5393 / 0.0803 / 0.0164 / 0.0127 | 0.2561 / 0.1236 / 0.1387 / 0.1385 | 1.000 | 0.905 | 0 |
| 8 | 4.0 | 3,194,880 | 0.4228 | 228.9 | 1.5605 / 0.0819 / 0.0167 / 0.0131 | 0.2705 / 0.1266 / 0.1382 / 0.1385 | 1.000 | 0.905 | 0 |
| 16 | 2.0 | 6,389,760 | 0.8421 | 228.8 | 1.5609 / 0.0845 / 0.0187 / 0.0139 | 0.2698 / 0.1281 / 0.1354 / 0.1365 | 1.000 | 0.905 | 0 |
| 32 | 1.0 | 12,779,520 | 1.6701 | 229.3 | 1.5677 / 0.0876 / 0.0205 / 0.0151 | 0.2708 / 0.1254 / 0.1329 / 0.1335 | 1.000 | 0.905 | 0 |

### 内容が × の質問

- r=2: 
  - 成績はいつ出ますか（正解の needs_review=True）: `{"answer": "提出期限は 9 月 